# Databricks Confluence RAG — Delta + Vector Search

Purpose: read Confluence pages, clean and chunk content, store RAG chunks in a Delta table, and prepare the data for Databricks Vector Search / AI Search.

Configure the Confluence credentials in a Databricks secret scope named `confluence`.


In [ ]:
import re
import hashlib
from datetime import datetime, timezone

import requests
from bs4 import BeautifulSoup
from pyspark.sql import functions as F


In [ ]:
CONFLUENCE_URL = dbutils.secrets.get("confluence", "url")
CONFLUENCE_EMAIL = dbutils.secrets.get("confluence", "email")
CONFLUENCE_API_TOKEN = dbutils.secrets.get("confluence", "api_token")

SPACE_KEY = "YOUR_SPACE_KEY"

session = requests.Session()
session.auth = (CONFLUENCE_EMAIL, CONFLUENCE_API_TOKEN)
session.headers.update({"Accept": "application/json"})

print("Confluence session created")


In [ ]:
def get_confluence_pages(space_key, limit=100):
    url = f"{CONFLUENCE_URL}/wiki/rest/api/content"
    params = {
        "spaceKey": space_key,
        "type": "page",
        "limit": limit,
        "expand": "body.storage,version,space"
    }

    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json().get("results", [])

pages = get_confluence_pages(SPACE_KEY)

print("Pages found:", len(pages))
for page in pages[:10]:
    print(page["id"], page["title"])


In [ ]:
def clean_html(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n", strip=True)
    return re.sub(r"\n+", "\n", text)

def chunk_text(text, chunk_size=1000, overlap=200):
    words = text.split()
    chunks = []
    start = 0
    step = max(1, chunk_size - overlap)

    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        start += step

    return chunks


In [ ]:
records = []

for page in pages:
    page_id = page["id"]
    title = page["title"]
    version = page.get("version", {}).get("number")

    html = page["body"]["storage"]["value"]
    content = clean_html(html)

    if not content.strip():
        continue

    content_hash = hashlib.sha256(
        content.encode("utf-8")
    ).hexdigest()

    page_url = f"{CONFLUENCE_URL}/wiki/pages/{page_id}"

    for i, chunk in enumerate(chunk_text(content)):
        records.append({
            "chunk_id": f"{page_id}_{content_hash[:12]}_{i}",
            "page_id": page_id,
            "title": title,
            "version": int(version) if version is not None else None,
            "chunk_number": i,
            "content": chunk,
            "content_hash": content_hash,
            "url": page_url,
            "ingested_at": datetime.now(timezone.utc).isoformat()
        })

print("Total RAG records:", len(records))


In [ ]:
df = spark.createDataFrame(records)

display(df)


In [ ]:
# Change these to your actual Unity Catalog names.
CATALOG = "your_catalog"
SCHEMA = "confluence_rag"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.gold_chunks"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(TABLE_NAME)
)

print("Saved:", TABLE_NAME)


In [ ]:
display(
    spark.table(TABLE_NAME)
    .select(
        "chunk_id",
        "page_id",
        "title",
        "version",
        "chunk_number",
        "content_hash",
        "url"
    )
)


## Vector Search / AI Search

Create a Databricks Vector Search endpoint and an index over the `gold_chunks` Delta table.

Recommended index columns:

- `chunk_id` — primary key
- `content` — searchable text
- `title`
- `page_id`
- `version`
- `url`
- `content_hash`

For production, use a Delta Sync Index so changes to the Delta table can flow into the search index.

The exact Vector Search API and endpoint configuration can vary by Databricks Runtime/workspace setup, so keep this notebook's ingestion layer independent from the search client.


## Production architecture

Confluence → Bronze → Silver → Gold Chunks → Databricks Vector Search / AI Search → Retriever → Evidence Gate → LLM → Answer + Confluence URL

Important rule: the LLM should answer only from retrieved Confluence evidence. If evidence is insufficient, return a controlled "not found in Confluence" response instead of guessing.
